# Homework #4: Model Building and Tracking

## Project Goal
In this project, I build a machine learning model using Formula 1 data and track all model runs using MLflow. My goal is to predict pit stop duration (`milliseconds`) using race and result-related features. I use a Random Forest Regressor because it supports tunable hyperparameters and works well for regression problems with structured tabular data.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [0]:
df_pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

In [0]:
display(df_pit)
display(df_races)
display(df_results)

In [0]:
df_model = (
    df_pit
    .join(
        df_races.select("raceId", "year", "round"),
        on="raceId",
        how="left"
    )
    .join(
        df_results.select("raceId", "driverId", "grid", "positionOrder", "constructorId"),
        on=["raceId", "driverId"],
        how="left"
    )
)

In [0]:
display(df_model)

In [0]:
df_model = (
    df_model
    .select(
        "milliseconds",
        "stop",
        "lap",
        "driverId",
        "raceId",
        "year",
        "round",
        "grid",
        "positionOrder",
        "constructorId"
    )
    .dropna()
)

In [0]:
display(df_model)

In [0]:
pdf = df_model.toPandas()
print(pdf.shape)
pdf.head()

In [0]:
X = pdf.drop(columns=["milliseconds"])
y = pdf["milliseconds"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Data Preparation

I use the `pit_stops` dataset as the main table because it contains the target variable, `milliseconds`, which represents pit stop duration. Then, I join the `races` dataset to add race-level features such as `year` and `round`, and the `results` dataset to add race result-related features such as `grid`, `positionOrder`, and `constructorId`.

After joining the tables, I keep only numeric variables and remove rows with missing values. I then convert the Spark dataframe into a pandas dataframe so I can use scikit-learn for model training. Finally, I split the data into training and testing sets.

In [0]:
mlflow.set_experiment("/Users/nl2994@columbia.edu/homework4_f1_mlflow")

In [0]:
def run_rf_experiment(n_estimators, max_depth, min_samples_split, min_samples_leaf, run_name):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mse = mean_squared_error(y_test, preds)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        # log parameters
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("min_samples_split", min_samples_split)
        mlflow.log_param("min_samples_leaf", min_samples_leaf)
        mlflow.log_param("random_state", 42)

        # log metrics
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        # log model
        mlflow.sklearn.log_model(model, "random_forest_model")

        # artifact 1: feature importance csv
        fi = pd.DataFrame({
            "feature": X_train.columns,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)

        fi_filename = f"feature_importance_{run_name}.csv"
        fi.to_csv(fi_filename, index=False)
        mlflow.log_artifact(fi_filename)

        # artifact 2: residual plot
        residuals = y_test - preds

        plt.figure(figsize=(8, 5))
        plt.scatter(preds, residuals, alpha=0.5)
        plt.axhline(0, linestyle="--")
        plt.xlabel("Predicted Pit Stop Duration")
        plt.ylabel("Residuals")
        plt.title(f"Residual Plot - {run_name}")
        plt.tight_layout()

        plot_filename = f"residual_plot_{run_name}.png"
        plt.savefig(plot_filename)
        plt.close()
        mlflow.log_artifact(plot_filename)

        print(f"Run completed: {run_name}")
        print(f"RMSE: {rmse:.3f}, MAE: {mae:.3f}, R2: {r2:.3f}")

In [0]:
run_rf_experiment(
    n_estimators=50,
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    run_name="test_run_1"
)

In [0]:
param_grid = [
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 50, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 50, "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 4, "min_samples_leaf": 2},
]

In [0]:
for i, params in enumerate(param_grid):
    run_rf_experiment(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        run_name=f"rf_run_{i+1}"
    )

## Model Choice and Tracking

I use a Random Forest Regressor to predict pit stop duration in milliseconds. I chose this model because it supports multiple tunable hyperparameters and performs well on structured tabular datasets.

For each run, I log the hyperparameters, the trained model, and all key regression metrics: MSE, RMSE, MAE, and R². In addition, I log at least two artifacts for each run: a feature importance CSV file and a residual plot. This allows me to compare multiple experiments and identify the best-performing model using MLflow.

## Best Model Selection

I chose **rf_run_8** as the best run because it had the lowest RMSE across all experiments. Since this is a regression problem, RMSE is a useful measure because it shows how far predictions are from the true values in the same unit as the target variable.

This run also produced a strong R² value, so it not only had lower error but also explained the data better than many of the other runs. Based on these results, `rf_run_8` is my final selected model.

The best run, `rf_run_8`, used `n_estimators = 100`, `max_depth = 15`, `min_samples_split = 4`, and `min_samples_leaf = 1`.

## Conclusion

In this project, I built a Random Forest regression model to predict Formula 1 pit stop duration using features from the `pit_stops`, `races`, and `results` datasets. I tracked each run with MLflow, including hyperparameters, model outputs, evaluation metrics, and artifacts.

After running 10 experiments with different hyperparameter settings, I selected `rf_run_8` as the best model because it achieved the lowest RMSE and a relatively strong R² value. This suggests that it provided the best balance of prediction accuracy among the models I tested.